In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 04 · Knowledge Retrieval (RAG) — ground every answer (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live` so this run never overwrites the pre-demo lab.

A member asks about a copay that changed this week — but facts change weekly, and you can't re-fine-tune for every formulary update. RAG grounds answers at inference time: embed the Sutter KB with `text-embedding-3-large`, retrieve the most relevant chunk by cosine similarity, and answer only from what was retrieved — with a citation. You get the exact value from `sutter_health_kb.md`, cited, with no retraining. *Ungrounded invents a copay; grounded quotes the source.*

---
## Step 1 — Config & client

*Load `.env` and build the `AzureOpenAI` client we'll reuse for every retrieval and chat call.*


In [ ]:
# Standard config + client. Same .env all labs share.
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv()
AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')

_cred = DefaultAzureCredential()
_tp   = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _tp,
    api_version    = AZURE_OPENAI_API_VERSION,
)
print('Client OK. Deployment:', BASE_DEPLOYMENT)


---
## Step 2 — Load and chunk the knowledge base

*Read `sutter_health_kb.md` and split it into bite-sized passages the retriever can score independently.*


In [ ]:
KB_FILE = Path('data/sutter_health_kb.md')
text = KB_FILE.read_text(encoding='utf-8')

# Naive but effective: split on H2/H3 headings, then on blank lines if a section is too big.
import re
sections = re.split(r'\n(?=##? )', text)
chunks = []
for s in sections:
    if len(s) <= 1200:
        chunks.append(s.strip())
    else:
        for para in s.split('\n\n'):
            if para.strip():
                chunks.append(para.strip())

print(f'KB chunks: {len(chunks)}')
print('\nFirst chunk (truncated):')
print(chunks[0][:300] + '...')


---
## Step 3 — Embed every chunk

*Turn each passage into a vector with `text-embedding-3-large` so we can compare it to the user's question by meaning, not keywords.*


In [ ]:
EMBED_DEPLOYMENT = os.environ.get('EMBED_DEPLOYMENT', 'text-embedding-3-large')

def embed(texts):
    r = client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts)
    return [d.embedding for d in r.data]

# Batch in chunks of 16 to stay friendly with rate limits
import numpy as np
vectors = []
for i in range(0, len(chunks), 16):
    vectors.extend(embed(chunks[i:i+16]))
vectors = np.array(vectors)
print(f'Embedded {len(vectors)} chunks, dim={vectors.shape[1]}')


---
## Step 4 — Retriever

*A tiny cosine-similarity retriever: embed the query, rank every chunk, return the top-k matches.*


In [ ]:
def retrieve(query, k=3):
    q = np.array(embed([query])[0])
    sims = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q) + 1e-9)
    top = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i]), chunks[i]) for i in top]

for idx, score, chunk in retrieve("Tier 1 generic 90-day mail order copay", k=3):
    print(f'[{idx:>3}] score={score:.3f}  {chunk[:120]}...')


---
## Step 5 — BEFORE: ungrounded answer

*Ask the base model **without any KB context** — watch it confidently invent a copay it has no way of knowing.*


In [ ]:
SYS = 'You are a Sutter Health Member Services voice agent. Be concise.'

PROMPT = "What is the Tier 1 generic 90-day mail-order copay at Sutter Health?"

resp = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[{'role':'system','content':SYS},{'role':'user','content':PROMPT}],
    temperature=0.0, max_tokens=200,
)
print('UNGROUNDED:')
print(resp.choices[0].message.content)


---
## Step 6 — AFTER: grounded answer with citations

*Stuff the top-k retrieved chunks into the system prompt and re-ask. Same question, but now the model answers from sources and cites them.*


In [ ]:
hits = retrieve(PROMPT, k=3)
context = '\n\n---\n\n'.join(f'[source {i+1}]\n{c}' for i, (_, _, c) in enumerate(hits))

GROUNDED_SYS = (
    'You are a Sutter Health Member Services voice agent. '
    'Answer ONLY using the sources below. If the answer is not in the sources, '
    'say "I do not have that information." Always cite the source number you used.'
    f'\n\nSOURCES:\n{context}'
)
resp = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[{'role':'system','content':GROUNDED_SYS},{'role':'user','content':PROMPT}],
    temperature=0.0, max_tokens=200,
)
print('GROUNDED:')
print(resp.choices[0].message.content)


---
## Step 7 — Refuse out-of-KB questions (no hallucination)

*Send a question that is **not** in the Sutter KB. The grounded model declines instead of hallucinating an answer.*


In [ ]:
OFF_TOPIC = "What's the cheapest flight from SFO to JFK tomorrow?"

print("=" * 70)
print("USER QUESTION (off-topic, not in the Sutter KB):")
print(f"  {OFF_TOPIC}")
print("=" * 70)

# Same RAG pipeline as Step 6 - retrieve, stuff, ask.
hits = retrieve(OFF_TOPIC, k=3)
context = '\n\n---\n\n'.join(f'[source {i+1}]\n{c}' for i, (_, _, c) in enumerate(hits))

print("\nTop retrieved chunks (note: none of them are about flights):")
for i, (idx, score, c) in enumerate(hits):
    print(f"  [source {i+1}] score={score:.3f}  {c[:80]}...")

resp = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[
        {'role':'system','content':GROUNDED_SYS.split('SOURCES:')[0] + f'SOURCES:\n{context}'},
        {'role':'user','content':OFF_TOPIC},
    ],
    temperature=0.0, max_tokens=120,
)
print("\nAGENT RESPONSE:")
print(f"  {resp.choices[0].message.content}")
print("\n=> The agent refuses cleanly instead of hallucinating a flight price.")


---
## Step 8 — Takeaway

*Wrap-up: when to reach for RAG vs. fine-tuning, and how they compose.*
